In [1]:
import cv2
import torch
import os
import sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from models.u_net import UNet
from models.load_model import load_model
from models.predict import predict
from utils.augmentation import get_val_transforms

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

device: cuda


In [3]:
model = load_model(model_path='/home/reva/trained UNet/original/best_model.pth', device=DEVICE)

/home/reva/G/skripsi/models/load_model.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location=device)


model has been loaded | Best Dice: 0.8773


In [ ]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd

IMG_DIR  = Path('/home/reva/G/skripsi/dataset/testing/images')
MASK_DIR = Path('/home/reva/G/skripsi/dataset/testing/masks')
SAVE_DIR = Path('/home/reva/G/skripsi/predict')

image_paths = sorted([p for p in IMG_DIR.iterdir() if p.suffix in ('.jpg', '.png', '.jpeg')])

all_metrics = []

for img_path in tqdm(image_paths, desc='Predicting'):
    stem      = img_path.stem                                          # e.g. ISIC_0012169
    mask_path = MASK_DIR / f'{stem}_segmentation.png'

    if not mask_path.exists():
        print(f'[SKIP] mask tidak ditemukan: {mask_path.name}')
        continue

    _, metrics = predict(
        model,
        img_path=str(img_path),
        mask_path=str(mask_path),
        save_path=str(SAVE_DIR),
        device=DEVICE,
    )
    metrics['image'] = img_path.name
    all_metrics.append(metrics)

df_metrics = pd.DataFrame(all_metrics).set_index('image')
print(f'\nSelesai: {len(df_metrics)} gambar diprediksi\n')
print(df_metrics.describe().loc[['mean', 'std', 'min', 'max']].to_string())

OutOfMemoryError: CUDA out of memory. Tried to allocate 736.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 353.31 MiB is free. Including non-PyTorch memory, this process has 3.19 GiB memory in use. Of the allocated memory 2.70 GiB is allocated by PyTorch, and 418.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
metric_cols = ['dice', 'iou', 'accuracy', 'precision', 'recall', 'specificity']
means = df_metrics[metric_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5CD']
bars = ax.bar(metric_cols, means.values, color=colors, width=0.55, edgecolor='white')

for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Rata-rata Metrik Segmentasi pada Testing Set', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('\nRingkasan metrik per gambar:')
display(df_metrics[metric_cols].style.background_gradient(cmap='RdYlGn', axis=0).format('{:.4f}'))